# Loop 3 · Optimizer: unsafe account help

After loop 2, the support team wanted faster account recovery. They added a section to AGENTS.md
telling the agent to "verify" customers by asking for their password or a 2FA code, then say it
has reset 2FA, and shipped it straight to production, without the gate.

`knowledge/account_access.md` only describes the self-service paths (backup codes, waiting out a
lockout, identity verification through support). The rule that support never asks for a
credential or changes an account from chat lives with the support leads, in their labels. This
loop moves it from their labels into AGENTS.md, where an edit like this one can't silently undo it.

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

New in this loop: a **must-pass** safety check, **judge alignment** (the generic `quality` judge
doesn't know Acme's account policy, so it is aligned to the reviewers' labels), and an
**optimizer** (MetaPrompt) writing the candidate. A person still promotes.

In [ ]:
import _session as s

experiment_id = s.setup(3)
r = s.results(3)

## 0. The change nobody gated

In [ ]:
from selfimprove import instructions

c = r["outside_change"]
print(f"v{c['version']} by {c['by']}: {c['message']}
")
print(instructions.diff(c["base_version"], c["version"]))

## 1. Find

In [ ]:
before = s.loop_traces(3, "before")
s.show(s.answers(before))

## 2. Signal: a must-pass check, and a judge that agrees with people

`no_credential_requests` is a regex check marked `must_pass: true`: a candidate that asks for a
credential or claims to turn off 2FA on **any** row can never ship, whatever else it improves.
`safe_account_help` is the same idea as a Guidelines judge, for phrasings no regex catches.

In [ ]:
s.show(s.verdicts(before))

The `quality` judge only sees the question and the answer. To it, "send me your 2FA code and I'll
reset it" can look helpful. **Align** measures its agreement with the reviewers on a held-out
slice of labeled traces and, if it is below the threshold, aligns it to their labels and
registers the aligned judge only if agreement improved.

In [ ]:
print(r["align"]["message"])
r["align"]["metrics"]

## 3. Store: triage and approve

In [ ]:
print(r["triage"]["message"])
print(r["approve"]["message"])
print(r["dataset"])

A person cleared the human queue before the optimizer ran, so its tests are in the train split:

In [ ]:
print(r["queue_review"])

## 4-5. Update with an optimizer, then gate

`optimize_prompts` rewrites AGENTS.md against every scorer on the train split (each evaluation is
a set of real OpenClaw runs). The holdout is never part of its training data. MetaPrompt goes
first; if the gate rejects its candidate, GEPA gets a turn. Every attempt is shown, including
the ones the gate stopped.

In [ ]:
for a in r["attempts"]:
    g = a.get("gate", {})
    print(f"{a['improver']:<11} {a['outcome']:<10} {a['message'][:90]}")
    print(f"{'':<11} gate: {g.get('outcome', '-')}  {g.get('reasons', '')[:120]}")

Both optimizers' first candidates were rejected. MetaPrompt's passed every quality rule and only
broke the **bloat** rule (`SI_MAX_PROMPT_GROWTH`, default 2.0): optimizers grow prompts, and that
rule is there to stop it. The limit belongs to a person. Here one read the diff, found a new
security section rather than padding, raised the limit for this agent, and re-ran the gate. The
decision is recorded with the run, and the gate still enforced every other rule:

In [ ]:
r.get("guardrail_change")

The re-run then failed on a must-pass row. Reading the row showed the **verifier** was wrong, not
the agent: a safe refusal ("Please don't send me your password…") matched the credential-request
regex. A must-pass check with a false positive blocks good candidates forever, so a person fixed
the pattern (refusals no longer match), confirmed no stored reviewer label changed, and re-ran the
gate. Verify the verifier before you trust its veto:

In [ ]:
r.get("verifier_fix")

In [ ]:
print(instructions.diff(r["production_before"], r["candidate"]))

Must-pass at 100% on both splits:

In [ ]:
s.show(s.gate_table(3), pct=True)

## 6. Measure

In [ ]:
s.show(s.before_after(3))

In [ ]:
s.show(s.answers(s.loop_traces(3, "after")))

**What moved:** the verifier itself is now partly learned (the aligned judge), and the candidate
comes from an optimizer. What stays with a person: the must-pass rule, the labels the judge is
aligned to, and the promotion.